In [19]:
!pip install pyspark

In [20]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("FraudDetection").getOrCreate()

In [21]:
data=[
    (1,"U1",2000,"Bangalore"),
    (2, "U2", 75000, "Mumbai"),
    (3, "U1", 300, "Bangalore"),
    (4, "U3", 120000, "Delhi"),
    (5, "U2", 500, "Mumbai"),
    (6, "U4", 90000, "Chennai"),
    (7, "U5", 1000, "Hyderabad"),
    (8, "U1", 60000, "Delhi"),
    (9, "U6", 1500, "Kolkata"),
    (10, "U7", 80000, "Mumbai"),
    (11, "U1", 250, "Bangalore"),
    (12, "U8", 150000, "Delhi"),
    (13, "U2", 700, "Mumbai"),
    (14, "U9", 95000, "Pune"),
    (15, "U10", 1200, "Hyderabad"),
    (16, "U3", 62000, "Delhi"),
    (17, "U11", 3000, "Bangalore"),
    (18, "U12", 85000, "Chennai"),
    (19, "U13", 400, "Mumbai"),
    (20, "U1", 1800, "Bangalore")
]

columns=["transaction_id","user_id","amount","location"]

df=spark.createDataFrame(data,columns)
df.show()

+--------------+-------+------+---------+
|transaction_id|user_id|amount| location|
+--------------+-------+------+---------+
|             1|     U1|  2000|Bangalore|
|             2|     U2| 75000|   Mumbai|
|             3|     U1|   300|Bangalore|
|             4|     U3|120000|    Delhi|
|             5|     U2|   500|   Mumbai|
|             6|     U4| 90000|  Chennai|
|             7|     U5|  1000|Hyderabad|
|             8|     U1| 60000|    Delhi|
|             9|     U6|  1500|  Kolkata|
|            10|     U7| 80000|   Mumbai|
|            11|     U1|   250|Bangalore|
|            12|     U8|150000|    Delhi|
|            13|     U2|   700|   Mumbai|
|            14|     U9| 95000|     Pune|
|            15|    U10|  1200|Hyderabad|
|            16|     U3| 62000|    Delhi|
|            17|    U11|  3000|Bangalore|
|            18|    U12| 85000|  Chennai|
|            19|    U13|   400|   Mumbai|
|            20|     U1|  1800|Bangalore|
+--------------+-------+------+---

In [22]:
from pyspark.sql.functions import count

user_txn_count = df.groupBy("user_id").agg(count("*").alias("txn_count"))

user_txn_count.show()

+-------+---------+
|user_id|txn_count|
+-------+---------+
|     U2|        3|
|     U4|        1|
|     U3|        2|
|     U6|        1|
|     U5|        1|
|     U1|        5|
|     U7|        1|
|    U10|        1|
|    U11|        1|
|    U12|        1|
|     U9|        1|
|     U8|        1|
|    U13|        1|
+-------+---------+



In [23]:
df = df.join(user_txn_count, on="user_id", how="left")
df.show()

+-------+--------------+------+---------+---------+
|user_id|transaction_id|amount| location|txn_count|
+-------+--------------+------+---------+---------+
|     U2|             2| 75000|   Mumbai|        3|
|     U2|             5|   500|   Mumbai|        3|
|     U4|             6| 90000|  Chennai|        1|
|     U3|             4|120000|    Delhi|        2|
|     U6|             9|  1500|  Kolkata|        1|
|     U5|             7|  1000|Hyderabad|        1|
|     U1|             1|  2000|Bangalore|        5|
|     U1|             3|   300|Bangalore|        5|
|     U1|             8| 60000|    Delhi|        5|
|     U7|            10| 80000|   Mumbai|        1|
|     U2|            13|   700|   Mumbai|        3|
|    U10|            15|  1200|Hyderabad|        1|
|    U11|            17|  3000|Bangalore|        1|
|     U3|            16| 62000|    Delhi|        2|
|    U12|            18| 85000|  Chennai|        1|
|     U9|            14| 95000|     Pune|        1|
|     U8|   

In [24]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "risk_score",
    when(col("amount") > 100000, 40)
    .when(col("amount") > 50000, 25)
    .otherwise(10)
)

df.show()



+-------+--------------+------+---------+---------+----------+
|user_id|transaction_id|amount| location|txn_count|risk_score|
+-------+--------------+------+---------+---------+----------+
|     U2|             2| 75000|   Mumbai|        3|        25|
|     U2|             5|   500|   Mumbai|        3|        10|
|     U4|             6| 90000|  Chennai|        1|        25|
|     U3|             4|120000|    Delhi|        2|        40|
|     U6|             9|  1500|  Kolkata|        1|        10|
|     U5|             7|  1000|Hyderabad|        1|        10|
|     U1|             1|  2000|Bangalore|        5|        10|
|     U1|             3|   300|Bangalore|        5|        10|
|     U1|             8| 60000|    Delhi|        5|        25|
|     U7|            10| 80000|   Mumbai|        1|        25|
|     U2|            13|   700|   Mumbai|        3|        10|
|    U10|            15|  1200|Hyderabad|        1|        10|
|    U11|            17|  3000|Bangalore|        1|    

In [25]:
df = df.withColumn(
    "risk_score",
    col("risk_score") +
    when(col("location").isin("Delhi", "Mumbai"), 15).otherwise(0)
)

df.show()

+-------+--------------+------+---------+---------+----------+
|user_id|transaction_id|amount| location|txn_count|risk_score|
+-------+--------------+------+---------+---------+----------+
|     U2|             2| 75000|   Mumbai|        3|        40|
|     U2|             5|   500|   Mumbai|        3|        25|
|     U4|             6| 90000|  Chennai|        1|        25|
|     U3|             4|120000|    Delhi|        2|        55|
|     U6|             9|  1500|  Kolkata|        1|        10|
|     U5|             7|  1000|Hyderabad|        1|        10|
|     U1|             1|  2000|Bangalore|        5|        10|
|     U1|             3|   300|Bangalore|        5|        10|
|     U1|             8| 60000|    Delhi|        5|        40|
|     U7|            10| 80000|   Mumbai|        1|        40|
|     U2|            13|   700|   Mumbai|        3|        25|
|    U10|            15|  1200|Hyderabad|        1|        10|
|    U11|            17|  3000|Bangalore|        1|    

In [26]:
df = df.withColumn(
    "risk_score",
    col("risk_score") +
    when(col("txn_count") > 2, 15).otherwise(0)
)

df.show()

+-------+--------------+------+---------+---------+----------+
|user_id|transaction_id|amount| location|txn_count|risk_score|
+-------+--------------+------+---------+---------+----------+
|     U2|             2| 75000|   Mumbai|        3|        55|
|     U2|             5|   500|   Mumbai|        3|        40|
|     U4|             6| 90000|  Chennai|        1|        25|
|     U3|             4|120000|    Delhi|        2|        55|
|     U6|             9|  1500|  Kolkata|        1|        10|
|     U5|             7|  1000|Hyderabad|        1|        10|
|     U1|             1|  2000|Bangalore|        5|        25|
|     U1|             3|   300|Bangalore|        5|        25|
|     U1|             8| 60000|    Delhi|        5|        55|
|     U7|            10| 80000|   Mumbai|        1|        40|
|     U2|            13|   700|   Mumbai|        3|        40|
|    U10|            15|  1200|Hyderabad|        1|        10|
|    U11|            17|  3000|Bangalore|        1|    

In [27]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col, when

window_spec = Window.partitionBy("user_id").orderBy("transaction_id")

df = df.withColumn(
    "prev_txn",
    lag("transaction_id").over(window_spec)
)

df = df.withColumn(
    "rapid_txn",
    when((col("transaction_id") - col("prev_txn") <= 5), 1).otherwise(0)
)

df.show()

+-------+--------------+------+---------+---------+----------+--------+---------+
|user_id|transaction_id|amount| location|txn_count|risk_score|prev_txn|rapid_txn|
+-------+--------------+------+---------+---------+----------+--------+---------+
|     U1|             1|  2000|Bangalore|        5|        25|    NULL|        0|
|     U1|             3|   300|Bangalore|        5|        25|       1|        1|
|     U1|             8| 60000|    Delhi|        5|        55|       3|        1|
|     U1|            11|   250|Bangalore|        5|        25|       8|        1|
|     U1|            20|  1800|Bangalore|        5|        25|      11|        0|
|    U10|            15|  1200|Hyderabad|        1|        10|    NULL|        0|
|    U11|            17|  3000|Bangalore|        1|        10|    NULL|        0|
|    U12|            18| 85000|  Chennai|        1|        25|    NULL|        0|
|    U13|            19|   400|   Mumbai|        1|        25|    NULL|        0|
|     U2|       

In [28]:
df = df.withColumn(
    "risk_score",
    col("risk_score") +
    when(col("rapid_txn") == 1, 20).otherwise(0)
)

df.show()

+-------+--------------+------+---------+---------+----------+--------+---------+
|user_id|transaction_id|amount| location|txn_count|risk_score|prev_txn|rapid_txn|
+-------+--------------+------+---------+---------+----------+--------+---------+
|     U1|             1|  2000|Bangalore|        5|        25|    NULL|        0|
|     U1|             3|   300|Bangalore|        5|        45|       1|        1|
|     U1|             8| 60000|    Delhi|        5|        75|       3|        1|
|     U1|            11|   250|Bangalore|        5|        45|       8|        1|
|     U1|            20|  1800|Bangalore|        5|        25|      11|        0|
|    U10|            15|  1200|Hyderabad|        1|        10|    NULL|        0|
|    U11|            17|  3000|Bangalore|        1|        10|    NULL|        0|
|    U12|            18| 85000|  Chennai|        1|        25|    NULL|        0|
|    U13|            19|   400|   Mumbai|        1|        25|    NULL|        0|
|     U2|       

In [29]:
df = df.withColumn(
    "final_fraud",
    when(col("risk_score") >= 60, 1).otherwise(0)
)

In [30]:
df.select(
    "user_id",
    "transaction_id",
    "amount",
    "location",
    "txn_count",
    "rapid_txn",
    "risk_score",
    "final_fraud"
).show()

+-------+--------------+------+---------+---------+---------+----------+-----------+
|user_id|transaction_id|amount| location|txn_count|rapid_txn|risk_score|final_fraud|
+-------+--------------+------+---------+---------+---------+----------+-----------+
|     U1|             1|  2000|Bangalore|        5|        0|        25|          0|
|     U1|             3|   300|Bangalore|        5|        1|        45|          0|
|     U1|             8| 60000|    Delhi|        5|        1|        75|          1|
|     U1|            11|   250|Bangalore|        5|        1|        45|          0|
|     U1|            20|  1800|Bangalore|        5|        0|        25|          0|
|    U10|            15|  1200|Hyderabad|        1|        0|        10|          0|
|    U11|            17|  3000|Bangalore|        1|        0|        10|          0|
|    U12|            18| 85000|  Chennai|        1|        0|        25|          0|
|    U13|            19|   400|   Mumbai|        1|        0|    

In [31]:
df.groupBy("final_fraud").count().show()

+-----------+-----+
|final_fraud|count|
+-----------+-----+
|          1|    2|
|          0|   18|
+-----------+-----+



In [32]:
df.orderBy(col("risk_score").desc()).show()

+-------+--------------+------+---------+---------+----------+--------+---------+-----------+
|user_id|transaction_id|amount| location|txn_count|risk_score|prev_txn|rapid_txn|final_fraud|
+-------+--------------+------+---------+---------+----------+--------+---------+-----------+
|     U1|             8| 60000|    Delhi|        5|        75|       3|        1|          1|
|     U2|             5|   500|   Mumbai|        3|        60|       2|        1|          1|
|     U2|             2| 75000|   Mumbai|        3|        55|    NULL|        0|          0|
|     U3|             4|120000|    Delhi|        2|        55|    NULL|        0|          0|
|     U8|            12|150000|    Delhi|        1|        55|    NULL|        0|          0|
|     U1|             3|   300|Bangalore|        5|        45|       1|        1|          0|
|     U1|            11|   250|Bangalore|        5|        45|       8|        1|          0|
|     U2|            13|   700|   Mumbai|        3|        4

In [34]:
df=df.drop("prev_txn")
df.show()

+-------+--------------+------+---------+---------+----------+---------+-----------+
|user_id|transaction_id|amount| location|txn_count|risk_score|rapid_txn|final_fraud|
+-------+--------------+------+---------+---------+----------+---------+-----------+
|     U1|             1|  2000|Bangalore|        5|        25|        0|          0|
|     U1|             3|   300|Bangalore|        5|        45|        1|          0|
|     U1|             8| 60000|    Delhi|        5|        75|        1|          1|
|     U1|            11|   250|Bangalore|        5|        45|        1|          0|
|     U1|            20|  1800|Bangalore|        5|        25|        0|          0|
|    U10|            15|  1200|Hyderabad|        1|        10|        0|          0|
|    U11|            17|  3000|Bangalore|        1|        10|        0|          0|
|    U12|            18| 85000|  Chennai|        1|        25|        0|          0|
|    U13|            19|   400|   Mumbai|        1|        25|   

- high amt=high risk
- high risk location = additional risk
- frequent transacyions= behavioral risk
- rapid transactions = velocity risk